# Conhecendo a requests

## Primeira requisição

In [ ]:
import requests

In [ ]:
r = requests.get('https://api.github.com/events')

In [ ]:
r

In [ ]:
r.status_code

In [ ]:
r.url

In [ ]:
r.text

In [ ]:
r.json()

In [ ]:
r = requests.get('https://api.github.com/versions')

In [ ]:
r.status_code

In [ ]:
r.json()

In [ ]:
headers = {'X-GitHub-Api-Version': '2022-11-28'}

In [ ]:
r = requests.get('https://api.github.com/events', headers=headers)
r.status_code

In [ ]:
username = 'vitorlinsbinski'
url = f'https://api.github.com/users/{username}'

r = requests.get(url)
print(r.status_code)
response_json = r.json()
print(response_json)
print(r.url)

In [ ]:
print(f"""
Nome: {response_json.get('name')}
Nome de usuário: {response_json.get('login')}
Número de repositórios públicos: {response_json.get('public_repos')}
Data de criação da conta no GitHub: {response_json.get('created_at')}
""")

## Obtendo dados dos repositórios

In [ ]:
headers = {'X-GitHub-Api-Version': '2022-11-28'}

In [ ]:
api_base_url = 'https://api.github.com'
owner = 'amzn'
url = f'{api_base_url}/users/{owner}/repos'

In [ ]:
url

In [ ]:
response = requests.get(url, headers=headers, params={'sort': 'name'})
response.status_code

In [ ]:
response.json()

In [ ]:
len(response.json())

## Autenticação

In [ ]:
access_token = 'YOUR_GITHUB_ACCESS_TOKEN'

In [ ]:
headers = {'Authorization': f'Bearer {access_token}',
           'X-GitHub-Api-Version': '2022-11-28'}
headers

In [ ]:
url

In [ ]:
repos_list = []
for page_num in range(1,7):
    try: 
        response = requests.get(url, headers=headers, params={'page': page_num})
        repos_list.append(response.json())
    except:
        repos_list.append(None)
    
repos_list

In [ ]:
total_repos = 0
for page in repos_list:
    total_repos += len(page)

total_repos

### Desafio

In [ ]:
username = 'amzn'
url_seguidores = f'{api_base_url}/users/{username}/followers'
url_seguidores

In [ ]:
r_seguidores = requests.get(url_seguidores, headers=headers)
response.status_code

In [ ]:
seguidores = r_seguidores.json()

In [ ]:
len(seguidores)

In [ ]:
followers_list = []
page = 1

while True:
    try:
        seguidores_list_page = requests.get(url_seguidores, headers=headers, params={'per_page': 100, 'page': page}).json()
        
        if len(seguidores_list_page) == 0:
            break 
        
        followers_list.extend(seguidores_list_page)
    except Exception as e:
        print(f"Erro na página {page}: {e}")
        followers_list.append(None)  
        break  
    finally:
        page += 1

print(followers_list)

In [ ]:
len(followers_list)

## Transformando os dados

In [ ]:
repos_list

In [ ]:
repos_list[0][5]['name']

In [ ]:
repos_name = []

for page in repos_list:
    for repo in page:
        repos_name.append(repo.get('name'))

repos_name[:10]

In [ ]:
len(repos_name)

## Linguagem dos repositórios

In [ ]:
repos_list[0][8].get('language')

In [ ]:
repos_language = []

for page in repos_list:
    for repo in page:
        repos_language.append(repo.get('language'))

repos_language[:10]

In [ ]:
len(repos_language)

In [ ]:
repos = {
    'repository_name': repos_name,
    'language': repos_language
}

import pandas as pd

dados_amz = pd.DataFrame(repos)
dados_amz

In [ ]:
dados_amz.to_csv('../amazon.csv', index=False)

In [ ]:
followes_usernames = []
for follower in followers_list:
    name = follower.get('login')
    followes_usernames.append(name)

followes_usernames[:10]

## Criando repositório com POST

In [ ]:
api_base_url = 'https://api.github.com'
url = f'{api_base_url}/user/repos'

url

In [ ]:
data = {
    'name': 'linguagens-utilizadas',
    'description': 'Repositório com as linguagens de programação da Amazon',
    'private': True
}

response = requests.post(url, headers=headers, json=data)
response.status_code

## Formato do arquivo

In [ ]:
import base64

In [ ]:
with open('../amazon.csv', 'rb') as file:
    file_content = file.read()

encoded_content = base64.b64encode(file_content)

## Upload de arquivo com PUT

In [ ]:
api_base_url = 'https://api.github.com'
username = 'vitorlinsbinski'
repo = 'linguagens-utilizadas'
path = 'amazon.csv'

url = f'{api_base_url}/repos/{username}/{repo}/contents/{path}'
url

In [ ]:
data = {
    'message': 'Adicionando um novo arquivo',
    'content': encoded_content.decode('utf-8')
}

response = requests.put(url, headers=headers, json=data)
response.status_code

## Fork

In [ ]:
api_base_url = 'https://api.github.com'
owner = 'amzn'
repo = 'app-platform'

url = f'{api_base_url}/repos/{owner}/{repo}/forks'
url

In [ ]:
response = requests.post(url, headers=headers)
response.status_code